# Iterative Pseudo — Case1/2 Base Hard Negative Pool

이 notebook은 **75 cutoff를 사용하지 않는다.**

각 fold의 전체 Case1/Case2에서 앞으로 pseudo가 될 수 있는 known-side patent를 수집한다.

- Case1: known plaintiff patent → `src`이므로 기존 C3 A/B/C/D hard-pool 규칙을 그대로 적용
- Case2: known defendant patent → 최종 방향이 `plaintiff → defendant`이므로 known patent를 억지로 src로 뒤집지 않고 별도 record로 저장
- real C3/past positive는 negative 후보에서 제외
- iterative pseudo positive는 아직 모르므로 최초 base pool에서는 제외하지 않음
- 이후 round에서는 `available_pool[src] = base_pool[src] - cumulative_positive_by_src[src]`

In [1]:
from pathlib import Path
import sys, pickle, json
import numpy as np
import pandas as pd

ROOT = Path("/home2/jmbyeon/developments/workspace/apps/uspto_litigation/소송예측모델 개발/소송예측9월/0918PSEUDO_ITERATIVE")
assert ROOT.exists(), ROOT
sys.path.insert(0, str(ROOT))

import config
from data import load_snapshot

OUT_DIR = ROOT / "fixed_data" / "base_hard_pools"
C3_DIR = ROOT / "fixed_data" / "c3_pools"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FOLDS = config.TRAIN_FOLDS

print("ROOT:", ROOT)
print("SNAP_DIR:", config.SNAP_DIR)
print("FOLDS:", FOLDS)

ROOT: /home2/jmbyeon/developments/workspace/apps/uspto_litigation/소송예측모델 개발/소송예측9월/0918PSEUDO_ITERATIVE
SNAP_DIR: /home2/jmbyeon/developments/workspace/apps/uspto_litigation/소송예측모델 개발/소송예측9월/0916PSEUDO_ONESHOT/snapshots_clean_0917
FOLDS: [(2004, 2005, 2006), (2006, 2007, 2008), (2008, 2009, 2010), (2010, 2011, 2012)]


## 1. 실제 Case1/Case2 구조 확인

먼저 snapshot에서 case type과 patent edge가 예상과 같은지 확인한다.

In [2]:
snap = load_snapshot(config.SNAP_DIR, 2006)

print("case_meta columns:")
print(snap["case_meta"].columns.tolist())

print("\ncase_type counts:")
print(snap["case_meta"]["case_type"].astype(str).value_counts(dropna=False))

print("\nRelevant edge types:")
for et in snap["data"].edge_types:
    if "case" in et or "patent" in et:
        print(et)

display(
    snap["case_meta"][
        snap["case_meta"]["case_type"].astype(str).isin(["case1", "case2"])
    ].head(10)
)

case_meta columns:
['case_row_id', 'node_idx', 'date_filed', 'case_year', 'case_type', 'plt_assignee_id', 'def_assignee_id', 'plt_patents', 'def_patents', 'total_patents']

case_type counts:
case_type
case1    2319
case2     411
case3      56
Name: count, dtype: int64

Relevant edge types:
('patent', 'cites', 'patent')
('patent', 'cited_by', 'patent')
('patent', 'has_ipc', 'ipc')
('ipc', 'rev_has_ipc', 'patent')
('patent', 'owns', 'company')
('company', 'rev_owns', 'patent')
('patent', 'prev_owns', 'company')
('company', 'rev_prev_owns', 'patent')
('case', 'plaintiff', 'company')
('company', 'rev_plaintiff', 'case')
('case', 'defendant', 'company')
('company', 'rev_defendant', 'case')
('case', 'plaintiff_patent', 'patent')
('patent', 'rev_plaintiff_patent', 'case')
('case', 'defendant_patent', 'patent')
('patent', 'rev_defendant_patent', 'case')
('patent', 'sue', 'patent')
('patent', 'sued_by', 'patent')


,case_row_id,node_idx,date_filed,case_year,case_type,plt_assignee_id,def_assignee_id,plt_patents,def_patents,total_patents
0,731,0,2003-02-12,2003,case1,4bf9ac33-5702-482e-be96-643d51986da5,118565fb-5037-4fb5-b31c-a1c6e7674c02,10,0,10
1,733,1,2003-02-25,2003,case1,c4ac72cb-1298-4554-a05a-d032650c5aee,81f0f642-5e68-4b71-8e1a-70b463bdfe5e,1,0,1
2,736,2,2003-03-07,2003,case1,2c7913cb-9e7e-4843-97e6-4a581a5c53a7,2d50fef7-9612-4706-a13c-d1bc81f0e317,2,0,2
3,738,3,2003-03-17,2003,case1,e9f547cd-3715-4846-932c-4568a567cf23,c39ffb46-c2f0-41aa-ab4c-11826d8f872d,1,0,1
4,740,4,2003-04-08,2003,case1,0e13ac0f-0985-402f-a546-f38a3dd82a15,ca075913-a758-4eee-ba07-2ccc45123416,1,0,1
5,744,5,2003-04-30,2003,case1,4bf9ac33-5702-482e-be96-643d51986da5,d06ae83d-336a-41bf-9045-85ec4d328c68,1,0,1
6,748,6,2003-05-12,2003,case1,9d514391-f7be-4da7-a917-095c377c3ee9,ab0006a9-9c47-4890-9f25-8e59d7d06432,1,0,1
7,749,7,2003-05-14,2003,case1,2c7913cb-9e7e-4843-97e6-4a581a5c53a7,2d50fef7-9612-4706-a13c-d1bc81f0e317,1,0,1
8,752,8,2003-05-23,2003,case1,2c7913cb-9e7e-4843-97e6-4a581a5c53a7,27eda0b7-fce0-49e5-b72b-33e1e564ab0d,1,0,1
9,753,9,2003-05-23,2003,case1,3c406e68-bfe9-4afd-a7ae-12d9abe41b19,00e96db4-87b6-4e6a-ac13-91109f4e278b,3,0,3


## 2. 기존 C3 hard-negative 규칙과 동일한 helper

In [3]:
def norm(x):
    if pd.isna(x): return None
    s = str(x)
    return s[:-2] if s.endswith(".0") else s

def relation_maps(obj):
    d = obj["data"]; p2ipc={}; ipc2p={}; adj={}
    if ("patent","has_ipc","ipc") in d.edge_types:
        for p,i in d["patent","has_ipc","ipc"].edge_index.cpu().t().tolist():
            p2ipc.setdefault(int(p),set()).add(int(i))
            ipc2p.setdefault(int(i),set()).add(int(p))
    for et in [("patent","cites","patent"),("patent","cited_by","patent")]:
        if et in d.edge_types:
            for s,t in d[et].edge_index.cpu().t().tolist():
                adj.setdefault(int(s),set()).add(int(t))
    return p2ipc, ipc2p, adj

def khop(adj, src, k=3):
    seen={int(src)}; frontier={int(src)}; layers=[]
    for _ in range(k):
        nxt=set()
        for x in frontier: nxt |= adj.get(x,set())
        nxt -= seen
        layers.append(nxt); seen |= nxt; frontier=nxt
        if not frontier:
            layers += [set()]*(k-len(layers)); break
    return layers

def ownership_groups():
    oc = pd.read_parquet(config.OWNERSHIP_PATH).copy()
    ren={}
    if "patent" in oc.columns and "patent_id" not in oc.columns: ren["patent"]="patent_id"
    if "assignee" in oc.columns and "assignee_id" not in oc.columns: ren["assignee"]="assignee_id"
    oc=oc.rename(columns=ren)
    oc["patent_id"]=oc["patent_id"].map(norm)
    oc["assignee_id"]=oc["assignee_id"].map(norm)
    oc["start_dt"]=pd.to_datetime(oc["start_dt"],errors="coerce")
    oc["end_dt"]=pd.to_datetime(oc["end_dt"],errors="coerce")
    return {str(a):g for a,g in oc.groupby("assignee_id",sort=False)}

def portfolio(groups, aid, cutoff_dt, p2i):
    aid=norm(aid)
    if aid is None or aid not in groups: return set()
    q=groups[aid]
    q=q[(q.start_dt.isna()|q.start_dt.le(cutoff_dt)) &
        (q.end_dt.isna()|q.end_dt.gt(cutoff_dt))]
    return {p2i[str(p)] for p in q.patent_id.dropna().astype(str).unique()
            if str(p) in p2i}

def load_real_positive_pairs(cutoff):
    path=C3_DIR/f"cutoff_{cutoff}"/"c3_data.pkl"
    assert path.exists(), path
    with open(path,"rb") as f:
        c3=pickle.load(f)
    return {(int(s),int(d)) for s,d in c3["all_positive_pairs"]}

## 3. 전체 Case1/Case2 known-side patent 추출

Case1은 plaintiff patent가 known, Case2는 defendant patent가 known이라는 현재 case 정의를 사용한다.

In [4]:
def extract_case12_candidates(base, future, ys, ye):
    cm=future["case_meta"].copy()
    cm["date_filed"]=pd.to_datetime(cm["date_filed"],errors="coerce")
    cm=cm[
        cm["case_type"].astype(str).isin(["case1","case2"]) &
        cm["date_filed"].dt.year.between(ys,ye)
    ].copy()

    p2i={str(k):int(v) for k,v in base["pat2idx"].items()}
    fpat=[str(x) for x in future["pat_list"]]
    d=future["data"]
    cplt={}; cdef={}

    for et,target in [
        (("case","plaintiff_patent","patent"),cplt),
        (("case","defendant_patent","patent"),cdef)
    ]:
        if et in d.edge_types:
            for c,p in d[et].edge_index.cpu().t().tolist():
                target.setdefault(int(c),set()).add(fpat[int(p)])

    rows=[]
    for _,r in cm.iterrows():
        ni=int(r["node_idx"]); ctype=str(r["case_type"])
        pl=sorted(x for x in cplt.get(ni,set()) if x in p2i)
        de=sorted(x for x in cdef.get(ni,set()) if x in p2i)

        if ctype=="case1":
            for p in pl:
                rows.append({
                    "case_row_id":int(r["case_row_id"]),
                    "case_type":"case1",
                    "date_filed":r["date_filed"],
                    "known_side":"plaintiff",
                    "known_patent":p,
                    "known_idx":p2i[p],
                    "candidate_assignee_id":r.get("def_assignee_id")
                })
        elif ctype=="case2":
            for p in de:
                rows.append({
                    "case_row_id":int(r["case_row_id"]),
                    "case_type":"case2",
                    "date_filed":r["date_filed"],
                    "known_side":"defendant",
                    "known_patent":p,
                    "known_idx":p2i[p],
                    "candidate_assignee_id":r.get("plt_assignee_id")
                })
    return pd.DataFrame(rows)

for cutoff,ys,ye in FOLDS:
    base=load_snapshot(config.SNAP_DIR,cutoff)
    future=load_snapshot(config.SNAP_DIR,ye)
    x=extract_case12_candidates(base,future,ys,ye)
    print("\n", "="*70)
    print(f"{cutoff}->{ys}-{ye}")
    print("rows:",len(x),
          "cases:",x["case_row_id"].nunique() if len(x) else 0,
          "known patents:",x["known_idx"].nunique() if len(x) else 0)
    if len(x):
        print(x["case_type"].value_counts().to_dict())
        display(x.head())


2004->2005-2006
rows: 2940 cases: 1401 known patents: 1957
{'case1': 2551, 'case2': 389}


,case_row_id,case_type,date_filed,known_side,known_patent,known_idx,candidate_assignee_id
0,929,case1,2005-01-18,plaintiff,6695516,444439,dac5a8d9-7c7b-4a5b-b71c-edd8901a79ca
1,931,case1,2005-01-25,plaintiff,4770320,20393,aef99c6f-1126-4128-a867-6c8cc515c271
2,932,case2,2005-01-31,defendant,6461429,389330,2c7913cb-9e7e-4843-97e6-4a581a5c53a7
3,932,case2,2005-01-31,defendant,6461685,389375,2c7913cb-9e7e-4843-97e6-4a581a5c53a7
4,936,case1,2005-02-10,plaintiff,5234251,100500,e5a9dc76-d416-456b-a44d-5967d3b1d531



2006->2007-2008
rows: 4232 cases: 1774 known patents: 2377
{'case1': 3623, 'case2': 609}


,case_row_id,case_type,date_filed,known_side,known_patent,known_idx,candidate_assignee_id
0,1135,case1,2007-02-08,plaintiff,5267651,85181,995aa0c1-0782-4147-b869-143325b0b4c6
1,1142,case1,2007-03-05,plaintiff,7128748,525377,e4e5846c-14a3-4bf9-afeb-7a66635bf88f
2,1145,case1,2007-03-14,plaintiff,5553486,147596,74303bb4-54da-4b2b-9367-2576a3204584
3,1145,case1,2007-03-14,plaintiff,6026677,262725,74303bb4-54da-4b2b-9367-2576a3204584
4,1148,case1,2007-03-27,plaintiff,7195053,541402,283f7b37-b8a2-4e58-aeff-60f536bc6387



2008->2009-2010
rows: 3740 cases: 1708 known patents: 2578
{'case1': 3188, 'case2': 552}


,case_row_id,case_type,date_filed,known_side,known_patent,known_idx,candidate_assignee_id
0,1270,case1,2009-01-14,plaintiff,6354119,314652,299d1876-e3c3-42d8-9e94-7d7cac3fb13a
1,1270,case1,2009-01-14,plaintiff,6868703,434493,299d1876-e3c3-42d8-9e94-7d7cac3fb13a
2,1271,case1,2009-01-16,plaintiff,6315524,305452,512afb97-0def-4d72-8551-f9244d28b7cf
3,1271,case1,2009-01-16,plaintiff,6390768,323174,512afb97-0def-4d72-8551-f9244d28b7cf
4,1271,case1,2009-01-16,plaintiff,6692234,393931,512afb97-0def-4d72-8551-f9244d28b7cf



2010->2011-2012
rows: 7211 cases: 3284 known patents: 3208
{'case1': 6468, 'case2': 743}


,case_row_id,case_type,date_filed,known_side,known_patent,known_idx,candidate_assignee_id
0,1422,case1,2011-01-06,plaintiff,6327812,274633,9cf615e2-0f26-4633-acad-88eec25203d1
1,1422,case1,2011-01-06,plaintiff,7690148,594728,9cf615e2-0f26-4633-acad-88eec25203d1
2,1423,case1,2011-01-11,plaintiff,5835897,153876,244fe9d9-2b4b-42f3-bbb1-fa706a55461c
3,1423,case1,2011-01-11,plaintiff,6370511,284755,244fe9d9-2b4b-42f3-bbb1-fa706a55461c
4,1423,case1,2011-01-11,plaintiff,7620560,578367,244fe9d9-2b4b-42f3-bbb1-fa706a55461c


## 4. Case1 src별 Base Hard Pool 생성

기존 C3와 동일한 A/B/C/D 정의:

- A = same IPC ∩ citation 1-hop
- B = same IPC ∩ citation ≤2-hop (A 제외)
- C = same IPC ∩ citation ≤3-hop (A/B 제외)
- D = 상대 회사 portfolio ∩ (same IPC ∪ citation ≤3-hop) (A/B/C 제외)

real positive는 제외한다. pseudo positive는 아직 제외하지 않는다.

In [5]:
def build_case1_pool(s, candidate_assignee_id, p2ipc, ipc2p, adj,
                     og, cutoff_dt, p2i, real_by_src):
    same=set()
    for ip in p2ipc.get(int(s),set()):
        same |= ipc2p.get(ip,set())

    layers=khop(adj,int(s),3)
    c1=layers[0] if len(layers)>0 else set()
    c2=c1|(layers[1] if len(layers)>1 else set())
    c3=c2|(layers[2] if len(layers)>2 else set())

    pf=portfolio(og,candidate_assignee_id,cutoff_dt,p2i)
    forbid={int(s)}|real_by_src.get(int(s),set())

    A=(same&c1)-forbid
    B=(same&c2)-forbid-A
    C=(same&c3)-forbid-A-B
    D=(pf&(same|c3))-forbid-A-B-C

    return A,B,C,D

In [6]:
def build_fold(cutoff,ys,ye):
    print("\n"+"="*90)
    print("BUILD",cutoff,"->",ys,ye)
    print("="*90)

    base=load_snapshot(config.SNAP_DIR,cutoff)
    future=load_snapshot(config.SNAP_DIR,ye)
    p2i={str(k):int(v) for k,v in base["pat2idx"].items()}
    p2ipc,ipc2p,adj=relation_maps(base)
    og=ownership_groups()
    cutoff_dt=pd.Timestamp(f"{cutoff}-12-30")

    real_pos=load_real_positive_pairs(cutoff)
    real_by_src={}
    for s,d in real_pos:
        real_by_src.setdefault(s,set()).add(d)

    cand=extract_case12_candidates(base,future,ys,ye)
    c1=cand[cand["case_type"]=="case1"].copy()
    c2=cand[cand["case_type"]=="case2"].copy()

    pools={}
    for _,r in c1.iterrows():
        s=int(r["known_idx"])
        A,B,C,D=build_case1_pool(
            s,r["candidate_assignee_id"],p2ipc,ipc2p,adj,
            og,cutoff_dt,p2i,real_by_src
        )
        o=pools.setdefault(s,{
            "A":set(),"B":set(),"C":set(),"D":set(),
            "hard_pool":set(),"case_row_ids":set()
        })
        o["A"]|=A; o["B"]|=B; o["C"]|=C; o["D"]|=D
        o["hard_pool"]|=(A|B|C|D)
        o["case_row_ids"].add(int(r["case_row_id"]))

    serial={}
    stats=[]
    for s,o in pools.items():
        serial[s]={k:sorted(v) for k,v in o.items()}
        stats.append({
            "src":s,
            "n_cases":len(o["case_row_ids"]),
            "n_A":len(o["A"]),"n_B":len(o["B"]),
            "n_C":len(o["C"]),"n_D":len(o["D"]),
            "n_hard":len(o["hard_pool"])
        })

    payload={
        "cutoff":cutoff,
        "target":[ys,ye],
        "case1_src_pools":serial,
        # Case2는 known patent가 dst이므로 방향을 뒤집지 않고 record 보존
        "case2_records":c2.to_dict("records"),
        "all_case12_records":cand.to_dict("records"),
        "real_positive_pairs":real_pos,
        "num_patents":len(p2i),
        "pseudo_positive_removed":False
    }

    path=OUT_DIR/f"pool_{cutoff}.pkl"
    with open(path,"wb") as f:
        pickle.dump(payload,f)

    sdf=pd.DataFrame(stats)
    sdf.to_csv(OUT_DIR/f"pool_{cutoff}_stats.csv",index=False)

    summary={
        "cutoff":cutoff,
        "target":f"{ys}-{ye}",
        "case12_rows":len(cand),
        "case12_cases":cand["case_row_id"].nunique() if len(cand) else 0,
        "case1_rows":len(c1),
        "case1_cases":c1["case_row_id"].nunique() if len(c1) else 0,
        "case1_unique_src":len(pools),
        "case2_rows":len(c2),
        "case2_cases":c2["case_row_id"].nunique() if len(c2) else 0,
        "case2_unique_known_dst":c2["known_idx"].nunique() if len(c2) else 0,
        "hard_avg":float(sdf["n_hard"].mean()) if len(sdf) else 0,
        "hard_median":float(sdf["n_hard"].median()) if len(sdf) else 0,
        "real_positive_pairs":len(real_pos)
    }
    (OUT_DIR/f"pool_{cutoff}_summary.json").write_text(
        json.dumps(summary,indent=2,default=str),encoding="utf-8"
    )
    print(json.dumps(summary,indent=2))
    return summary

In [ ]:
# ============================================================
# 왜 1401 cases만 남는지 단계별 검증
# ============================================================

cutoff, ys, ye = 2004, 2005, 2006

base = load_snapshot(config.SNAP_DIR, cutoff)
future = load_snapshot(config.SNAP_DIR, ye)

cm = future["case_meta"].copy()
cm["date_filed"] = pd.to_datetime(cm["date_filed"], errors="coerce")

# ------------------------------------------------------------
# 1. 2005~2006 전체 Case1/2
# ------------------------------------------------------------
target = cm[
    cm["case_type"].astype(str).isin(["case1", "case2"])
    & cm["date_filed"].dt.year.between(ys, ye)
].copy()

print("=" * 80)
print("1. 2005~2006 Case1/2 전체")
print("=" * 80)

print("cases:", target["case_row_id"].nunique())
print(target["case_type"].astype(str).value_counts())


# ------------------------------------------------------------
# 2. snapshot 2006에서 known patent 연결 확인
#    여기서는 아직 snapshot 2004 존재 여부 필터 안 함
# ------------------------------------------------------------

future_pat = [str(x) for x in future["pat_list"]]
d = future["data"]

cplt = {}
cdef = {}

et_plt = ("case", "plaintiff_patent", "patent")
et_def = ("case", "defendant_patent", "patent")

if et_plt in d.edge_types:
    for c, p in d[et_plt].edge_index.cpu().t().tolist():
        cplt.setdefault(int(c), set()).add(future_pat[int(p)])

if et_def in d.edge_types:
    for c, p in d[et_def].edge_index.cpu().t().tolist():
        cdef.setdefault(int(c), set()).add(future_pat[int(p)])


rows_before = []

for _, r in target.iterrows():

    ni = int(r["node_idx"])
    ctype = str(r["case_type"])

    # ★ 2004 필터 없음
    if ctype == "case1":
        pats = sorted(cplt.get(ni, set()))
    else:
        pats = sorted(cdef.get(ni, set()))

    for p in pats:
        rows_before.append({
            "case_row_id": int(r["case_row_id"]),
            "case_type": ctype,
            "patent": p
        })

before = pd.DataFrame(rows_before)

print("\n" + "=" * 80)
print("2. known patent 연결 후 — 2004 존재 여부 필터 전")
print("=" * 80)

print("rows:", len(before))
print("cases:", before["case_row_id"].nunique())
print("known patents:", before["patent"].nunique())
print(before["case_type"].value_counts())


# ------------------------------------------------------------
# 3. snapshot 2004에 존재하는 patent만 남김
# ------------------------------------------------------------

base_patents = set(str(k) for k in base["pat2idx"].keys())

after = before[
    before["patent"].isin(base_patents)
].copy()

print("\n" + "=" * 80)
print("3. snapshot 2004 patent 존재 필터 후")
print("=" * 80)

print("rows:", len(after))
print("cases:", after["case_row_id"].nunique())
print("known patents:", after["patent"].nunique())
print(after["case_type"].value_counts())


# ------------------------------------------------------------
# 4. 정확히 얼마나 빠졌는지
# ------------------------------------------------------------

before_cases = set(before["case_row_id"])
after_cases = set(after["case_row_id"])

lost_cases = before_cases - after_cases

print("\n" + "=" * 80)
print("4. 2004 snapshot 필터 영향")
print("=" * 80)

print("patent rows before :", len(before))
print("patent rows after  :", len(after))
print("patent rows removed:", len(before) - len(after))

print()

print("cases before       :", len(before_cases))
print("cases after        :", len(after_cases))
print("cases completely removed:", len(lost_cases))

print()

print(
    "case retention:",
    f"{len(after_cases) / len(before_cases) * 100:.2f}%"
    if before_cases else "N/A"
)


# ------------------------------------------------------------
# 5. Case1 / Case2 각각 몇 case가 빠졌는지
# ------------------------------------------------------------

case_compare = (
    before[["case_row_id", "case_type"]]
    .drop_duplicates()
    .assign(
        survived=lambda x:
            x["case_row_id"].isin(after["case_row_id"])
    )
)

print("\n" + "=" * 80)
print("5. Case type별 case 생존 여부")
print("=" * 80)

display(
    pd.crosstab(
        case_compare["case_type"],
        case_compare["survived"],
        margins=True
    )
)


# ------------------------------------------------------------
# 6. 현재 notebook의 1401과 정확히 같은지
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("6. 현재 notebook 결과와 비교")
print("=" * 80)

print("expected current notebook cases = 1401")
print("recomputed after-filter cases   =", after["case_row_id"].nunique())

assert after["case_row_id"].nunique() == 1401, \
    "현재 notebook의 1401과 재계산 결과가 다름"

print("✓ 1401 case 재현됨")

## 5. 전체 fold 생성

In [7]:
summaries=[]
for cutoff,ys,ye in FOLDS:
    summaries.append(build_fold(cutoff,ys,ye))

summary_df=pd.DataFrame(summaries)
display(summary_df)
summary_df.to_csv(OUT_DIR/"base_pool_summary.csv",index=False)


BUILD 2004 -> 2005 2006
{
  "cutoff": 2004,
  "target": "2005-2006",
  "case12_rows": 2940,
  "case12_cases": 1401,
  "case1_rows": 2551,
  "case1_cases": 1214,
  "case1_unique_src": 1746,
  "case2_rows": 389,
  "case2_cases": 187,
  "case2_unique_known_dst": 351,
  "hard_avg": 1646.320160366552,
  "hard_median": 184.5,
  "real_positive_pairs": 187
}

BUILD 2006 -> 2007 2008
{
  "cutoff": 2006,
  "target": "2007-2008",
  "case12_rows": 4232,
  "case12_cases": 1774,
  "case1_rows": 3623,
  "case1_cases": 1538,
  "case1_unique_src": 2095,
  "case2_rows": 609,
  "case2_cases": 236,
  "case2_unique_known_dst": 466,
  "hard_avg": 2252.7680190930787,
  "hard_median": 340.0,
  "real_positive_pairs": 295
}

BUILD 2008 -> 2009 2010
{
  "cutoff": 2008,
  "target": "2009-2010",
  "case12_rows": 3740,
  "case12_cases": 1708,
  "case1_rows": 3188,
  "case1_cases": 1483,
  "case1_unique_src": 2272,
  "case2_rows": 552,
  "case2_cases": 225,
  "case2_unique_known_dst": 469,
  "hard_avg": 2792.805897

,cutoff,target,case12_rows,case12_cases,case1_rows,case1_cases,case1_unique_src,case2_rows,case2_cases,case2_unique_known_dst,hard_avg,hard_median,real_positive_pairs
0,2004,2005-2006,2940,1401,2551,1214,1746,389,187,351,1646.320160,184.5,187
1,2006,2007-2008,4232,1774,3623,1538,2095,609,236,466,2252.768019,340.0,295
2,2008,2009-2010,3740,1708,3188,1483,2272,552,225,469,2792.805898,424.5,354
3,2010,2011-2012,7211,3284,6468,3023,2958,743,261,457,3818.789385,696.5,656


## 6. Real-positive collision 검증

In [8]:
rows=[]
for cutoff,ys,ye in FOLDS:
    with open(OUT_DIR/f"pool_{cutoff}.pkl","rb") as f:
        obj=pickle.load(f)

    real={(int(s),int(d)) for s,d in obj["real_positive_pairs"]}
    collision=set()
    empty=0

    for s,o in obj["case1_src_pools"].items():
        s=int(s)
        pool={int(x) for x in o["hard_pool"]}
        if not pool: empty+=1
        for d in pool:
            if (s,d) in real:
                collision.add((s,d))

    rows.append({
        "fold":f"{cutoff}->{ys}-{ye}",
        "case1_pool_src":len(obj["case1_src_pools"]),
        "case2_records":len(obj["case2_records"]),
        "real_positive_collision":len(collision),
        "empty_case1_pool":empty
    })

verify_df=pd.DataFrame(rows)
display(verify_df)

assert verify_df["real_positive_collision"].sum()==0
print("✓ REAL POSITIVE COLLISION = 0")
print("✓ BASE POOL BUILD PASSED")

,fold,case1_pool_src,case2_records,real_positive_collision,empty_case1_pool
0,2004->2005-2006,1746,389,0,32
1,2006->2007-2008,2095,609,0,33
2,2008->2009-2010,2272,552,0,32
3,2010->2011-2012,2958,743,0,37


✓ REAL POSITIVE COLLISION = 0
✓ BASE POOL BUILD PASSED


## 다음 단계

이 notebook을 실행한 뒤 `base_pool_summary.csv`와 마지막 verification 표를 확인한다.

그 다음에는:

1. Case2 방향을 보존하는 pseudo candidate/scoring 구현
2. Round 0 Case3-only 모델 준비
3. 현재 모델로 Case1/2 재추론
4. high-confidence pseudo 일부 추가
5. 누적 positive를 base pool에서 제외
6. 재학습 → fixed validation → 다음 round

순서로 진행한다.

In [9]:
# ============================================================
# Case1 known-src vs Case3 real-src overlap
# ============================================================

rows = []

for cutoff, ys, ye in FOLDS:

    with open(OUT_DIR / f"pool_{cutoff}.pkl", "rb") as f:
        obj = pickle.load(f)

    # Case1에서 현재 확정되어 있는 src
    case1_src = {
        int(s) for s in obj["case1_src_pools"].keys()
    }

    # Case3 real positive의 src
    case3_src = {
        int(s) for s, d in obj["real_positive_pairs"]
    }

    overlap = case1_src & case3_src

    rows.append({
        "fold": f"{cutoff}->{ys}-{ye}",

        "case1_unique_src": len(case1_src),
        "case3_unique_src": len(case3_src),

        "overlap_src": len(overlap),

        "case1_overlap_pct":
            len(overlap) / len(case1_src) * 100
            if case1_src else 0,

        "case3_overlap_pct":
            len(overlap) / len(case3_src) * 100
            if case3_src else 0,
    })

overlap_df = pd.DataFrame(rows)

display(overlap_df)

,fold,case1_unique_src,case3_unique_src,overlap_src,case1_overlap_pct,case3_overlap_pct
0,2004->2005-2006,1746,92,6,0.343643,6.521739
1,2006->2007-2008,2095,134,17,0.811456,12.686567
2,2008->2009-2010,2272,167,15,0.660211,8.982036
3,2010->2011-2012,2958,239,24,0.811359,10.041841
